# Physics-Informed Graph Neural Network for River Modeling (Delft3D FM)

This notebook trains a PIGNN to model river hydrodynamics based on the 2D Shallow Water Equations, utilizing a mesh generated by Delft3D FM. It forces the network strictly using real physical boundaries.

In [ ]:
# 1. Download Data directly from GitHub
!git clone https://github.com/ATIK2110018/gironde_PINN.git
%cd gironde_PINN

In [ ]:
# 2. Install required libraries
!pip install torch-geometric netCDF4 xarray matplotlib scipy

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
import netCDF4 as nc
import numpy as np
import math
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

# Setup GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ==========================================
# 1. DATA EXTRACTION FUNCTIONS
# ==========================================
def extract_graph_from_netcdf(nc_file_path):
    print(f"Reading mesh from {nc_file_path}...")
    dataset = nc.Dataset(nc_file_path, 'r')
    
    if 'mesh2d_node_x' in dataset.variables:
        node_x = dataset.variables['mesh2d_node_x'][:]
        node_y = dataset.variables['mesh2d_node_y'][:]
        edge_nodes = dataset.variables['mesh2d_edge_nodes'][:]
        node_z = dataset.variables['mesh2d_node_z'][:] if 'mesh2d_node_z' in dataset.variables else np.zeros_like(node_x)
    elif 'NetNode_x' in dataset.variables:
        node_x = dataset.variables['NetNode_x'][:]
        node_y = dataset.variables['NetNode_y'][:]
        edge_nodes = dataset.variables['NetLink'][:]
        node_z = dataset.variables['NetNode_z'][:] if 'NetNode_z' in dataset.variables else np.zeros_like(node_x)
    else:
        raise ValueError("Could not find standard node coordinate variables.")

    edge_index_list, edge_attr_list = [], []
    for i in range(edge_nodes.shape[1] if edge_nodes.shape[0] == 2 else edge_nodes.shape[0]):
        if edge_nodes.shape[0] == 2:
            n1 = int(edge_nodes[0, i]) - 1
            n2 = int(edge_nodes[1, i]) - 1
        else:
            n1 = int(edge_nodes[i, 0]) - 1
            n2 = int(edge_nodes[i, 1]) - 1
            
        if n1 >= 0 and n2 >= 0:
            edge_index_list.extend([[n1, n2], [n2, n1]])
            dx, dy = node_x[n2] - node_x[n1], node_y[n2] - node_y[n1]
            dist = math.sqrt(dx**2 + dy**2)
            edge_attr_list.extend([[dx, dy, dist], [-dx, -dy, dist]])
            
    edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr_list, dtype=torch.float32)
    node_coords = torch.tensor(np.column_stack((node_x, node_y)), dtype=torch.float32)
    node_z = torch.tensor(node_z, dtype=torch.float32)
    dataset.close()
    return node_coords, edge_index, edge_attr, node_z

def load_friction_xyz(filepath, node_coords):
    print(f"Loading friction from {filepath}...")
    data = np.loadtxt(filepath)
    val_interp = griddata((data[:, 0], data[:, 1]), data[:, 2], (node_coords[:, 0], node_coords[:, 1]), method='nearest')
    return torch.tensor(val_interp, dtype=torch.float32)

def load_boundary_pli(filepath, node_coords):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    coords = np.array([[float(p[0]), float(p[1])] for line in lines[2:] if len(p := line.strip().split()) >= 2])
    node_coords_np = node_coords.numpy()
    return list(set(np.argmin(np.sqrt(np.sum((node_coords_np - pt)**2, axis=1))) for pt in coords))

def load_boundary_bc(filepath):
    times, values = [], []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                try:
                    times.append(float(parts[0]))
                    values.append(float(parts[1]))
                except ValueError:
                    continue
    return np.array(times), np.array(values)

In [ ]:
# ==========================================
# 2. GRAPH NEURAL NETWORK ARCHITECTURE
# ==========================================
class HydroMPNNLayer(MessagePassing):
    def __init__(self, node_in_dim, edge_in_dim, out_dim):
        super(HydroMPNNLayer, self).__init__(aggr='mean')
        self.message_mlp = nn.Sequential(nn.Linear(node_in_dim * 2 + edge_in_dim, 64), nn.ReLU(), nn.Linear(64, 64))
        self.update_mlp = nn.Sequential(nn.Linear(node_in_dim + 64, 64), nn.ReLU(), nn.Linear(64, out_dim))
    def forward(self, x, edge_index, edge_attr): return self.propagate(edge_index, x=x, edge_attr=edge_attr)
    def message(self, x_i, x_j, edge_attr): return self.message_mlp(torch.cat([x_i, x_j, edge_attr], dim=1))
    def update(self, aggr_out, x): return self.update_mlp(torch.cat([x, aggr_out], dim=1))

class RiverPIGNN(nn.Module):
    def __init__(self, node_features_dim=5, hidden_dim=64, edge_dim=3):
        super(RiverPIGNN, self).__init__()
        self.encoder = nn.Linear(node_features_dim, hidden_dim)
        self.mpnn1 = HydroMPNNLayer(hidden_dim, edge_dim, hidden_dim)
        self.mpnn2 = HydroMPNNLayer(hidden_dim, edge_dim, hidden_dim)
        self.decoder = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Linear(32, 3))
    def forward(self, x, edge_index, edge_attr):
        h = F.relu(self.encoder(x))
        h = F.relu(self.mpnn1(h, edge_index, edge_attr))
        h = self.mpnn2(h, edge_index, edge_attr)
        return self.decoder(h) + x[:, :3]

In [ ]:
# ==========================================
# 3. PHYSICS LOSS 
# ==========================================
def physics_loss_swe(state_t, state_t_next, edge_index, edge_attr, dt=1.0, g=9.81):
    eta_t, u_t, v_t, zb, cf = state_t.T
    h_t = eta_t - zb
    eta_next, u_next, v_next = state_t_next.T
    deta_dt, du_dt, dv_dt = (eta_next - eta_t)/dt, (u_next - u_t)/dt, (v_next - v_t)/dt
    
    row, col = edge_index
    dx, dy, dist = edge_attr.T
    
    delta_eta = eta_t[col] - eta_t[row]
    delta_u = u_t[col] - u_t[row]
    delta_v = v_t[col] - v_t[row]
    delta_hu = (h_t[col] * u_t[col]) - (h_t[row] * u_t[row])
    delta_hv = (h_t[col] * v_t[col]) - (h_t[row] * v_t[row])
    
    weight_x, weight_y = dx / (dist ** 2 + 1e-8), dy / (dist ** 2 + 1e-8)
    num_nodes = eta_t.size(0)
    
    def scatter_add_pt(src, index, dim_size):
        return torch.zeros(dim_size, dtype=src.dtype, device=src.device).scatter_add_(0, index, src)
    
    grad_eta_x, grad_eta_y = scatter_add_pt(delta_eta * weight_x, row, num_nodes), scatter_add_pt(delta_eta * weight_y, row, num_nodes)
    grad_u_x, grad_u_y = scatter_add_pt(delta_u * weight_x, row, num_nodes), scatter_add_pt(delta_u * weight_y, row, num_nodes)
    grad_v_x, grad_v_y = scatter_add_pt(delta_v * weight_x, row, num_nodes), scatter_add_pt(delta_v * weight_y, row, num_nodes)
    grad_hu_x, grad_hv_y = scatter_add_pt(delta_hu * weight_x, row, num_nodes), scatter_add_pt(delta_hv * weight_y, row, num_nodes)
    
    mass_residual = deta_dt + grad_hu_x + grad_hv_y
    velocity_magnitude = torch.sqrt(u_t**2 + v_t**2 + 1e-8)
    momentum_x_residual = du_dt + u_t * grad_u_x + v_t * grad_u_y + g * grad_eta_x + cf * u_t * velocity_magnitude / (h_t + 1e-8)
    momentum_y_residual = dv_dt + u_t * grad_v_x + v_t * grad_v_y + g * grad_eta_y + cf * v_t * velocity_magnitude / (h_t + 1e-8)
    
    return torch.mean(mass_residual**2) + torch.mean(momentum_x_residual**2) + torch.mean(momentum_y_residual**2)

### Parse Real Data and Boundaries

In [ ]:
netcdf_path = "../data/input/FlowFM_net.nc"
node_coords, edge_index, edge_attr, node_z = extract_graph_from_netcdf(netcdf_path)
num_nodes = node_coords.size(0)

# Load real friction map
friction = load_friction_xyz("../data/input/frictioncoefficient.xyz", node_coords)

# Load Boundary Nodes
bnd_port = load_boundary_pli("../data/input/port block.pli", node_coords)
bnd_garonne = load_boundary_pli("../data/input/garonne.pli", node_coords)
bnd_dordogne = load_boundary_pli("../data/input/dordogne.pli", node_coords)

# Load Time Series (Water Level & Discharges)
t_port, v_port = load_boundary_bc("../data/input/WaterLevel.bc")
t_garonne, v_garonne = load_boundary_bc("../data/input/garonne.bc")
t_dordogne, v_dordogne = load_boundary_bc("../data/input/dordogne.bc")

print(f"Successfully parsed fully realistic setup!")
print(f"Port Boundary Nodes: {len(bnd_port)}")
print(f"Garonne Boundary Nodes: {len(bnd_garonne)}")
print(f"Dordogne Boundary Nodes: {len(bnd_dordogne)}")

### Pre-Training Visualizations

In [ ]:
triang = mtri.Triangulation(node_coords[:, 0].numpy(), node_coords[:, 1].numpy())

plt.figure(figsize=(18, 5))
plt.subplot(1, 3, 1)
plt.triplot(triang, 'k-', lw=0.1, alpha=0.5)
plt.scatter(node_coords[bnd_port, 0], node_coords[bnd_port, 1], c='red', s=10, label='Port (Water Level)')
plt.scatter(node_coords[bnd_garonne, 0], node_coords[bnd_garonne, 1], c='green', s=10, label='Garonne (Discharge)')
plt.scatter(node_coords[bnd_dordogne, 0], node_coords[bnd_dordogne, 1], c='orange', s=10, label='Dordogne (Discharge)')
plt.title("Mesh & Forced Boundaries")
plt.legend()
plt.axis('equal')

plt.subplot(1, 3, 2)
plt.tripcolor(triang, node_z.numpy(), cmap='terrain')
plt.colorbar(label='Bed Level (m)')
plt.title("River Bathymetry")
plt.axis('equal')

plt.subplot(1, 3, 3)
plt.tripcolor(triang, friction.numpy(), cmap='YlOrRd')
plt.colorbar(label="Manning's n")
plt.title("Friction Coefficient from .xyz")
plt.axis('equal')
plt.tight_layout()
plt.show()

### Training Loop with Physics Loss, Boundary Loss, and Data Loss
A true PINN optimizes a combination of PDE residuals (Physics), Boundary adherence (Boundary), and Ground Truth matching (Data).

In [ ]:
node_coords = node_coords.to(device)
edge_index = edge_index.to(device)
edge_attr = edge_attr.to(device)
node_z = node_z.to(device)
friction = friction.to(device)

model = RiverPIGNN(node_features_dim=5, hidden_dim=64, edge_dim=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Simulation Parameters
dt = 3600.0  # Time step matches BC files (1 hour)
max_time = t_port[-1] if len(t_port) > 0 else 36000
current_time = 0.0

# Initial State Setup (Flat water surface 2m above highest bed point to prevent dry cells initially)
initial_water_level = float(torch.max(node_z)) + 2.0
state_t = torch.zeros((num_nodes, 5), dtype=torch.float32).to(device)
state_t[:, 0] = initial_water_level  # eta
state_t[:, 3] = node_z               # bathymetry
state_t[:, 4] = friction             # friction

def get_interp_val(t, times, values):
    return np.interp(t, times, values)

print("Starting Dynamic Physics Solver...")
while current_time < max_time:
    model.train()
    optimizer.zero_grad()

    # Note: We do NOT hardcode the boundary conditions directly into the state_t.
    # Instead, we let the network predict the whole field, and penalize it using Boundary Loss!
    
    # 1. Predict next state t+dt
    predicted_state_next = model(state_t, edge_index, edge_attr)
    
    # 2. Compute Physics Loss (Ensures SWE mass & momentum conservation)
    loss_physics = physics_loss_swe(state_t, predicted_state_next, edge_index, edge_attr, dt=dt)
    
    # 3. Compute Boundary Loss (Ensures prediction at boundaries matches the .bc files)
    target_port_wl = get_interp_val(current_time + dt, t_port, v_port)
    target_gar_q = get_interp_val(current_time + dt, t_garonne, v_garonne) * 0.001
    target_dor_q = get_interp_val(current_time + dt, t_dordogne, v_dordogne) * 0.001
    
    # MSE between predicted values at boundary nodes and the true boundary targets
    loss_bnd_port = F.mse_loss(predicted_state_next[bnd_port, 0], torch.tensor(target_port_wl, dtype=torch.float32, device=device).expand(len(bnd_port)))
    loss_bnd_gar = F.mse_loss(predicted_state_next[bnd_garonne, 1], torch.tensor(target_gar_q, dtype=torch.float32, device=device).expand(len(bnd_garonne)))
    loss_bnd_dor = F.mse_loss(predicted_state_next[bnd_dordogne, 1], torch.tensor(target_dor_q, dtype=torch.float32, device=device).expand(len(bnd_dordogne)))
    
    loss_boundary = loss_bnd_port + loss_bnd_gar + loss_bnd_dor
    
    # 4. Compute Data Loss (Placeholder for now until true FlowFM_map output is loaded)
    loss_data = torch.tensor(0.0, device=device)
    
    # 5. Total Combined Loss
    # You can tune weights (e.g., heavily weighting the boundary loss so the network respects it)
    total_loss = (1.0 * loss_physics) + (100.0 * loss_boundary) + (0.0 * loss_data)
    
    total_loss.backward()
    optimizer.step()
    
    if current_time % (3600*4) == 0: # Print every 4 hours
        print(f"Time: {current_time/3600:.1f}h | Total: {total_loss.item():.4f} | Phys: {loss_physics.item():.4f} | Bnd: {loss_boundary.item():.4f} | Data: {loss_data.item():.4f}")
        
    # Step time forward (Auto-regressive roll out)
    # We construct a new state tensor for t+1 to ensure shape [num_nodes, 5]
    state_t = torch.zeros((num_nodes, 5), dtype=torch.float32, device=device)
    state_t[:, :3] = predicted_state_next.detach().clone()  # Copy [eta, u, v]
    state_t[:, 3] = node_z                                  # Re-apply static bathymetry
    state_t[:, 4] = friction                                # Re-apply static friction
    
    current_time += dt

print("Simulation Complete!")
torch.save(model.state_dict(), "pignn_weights_sim.pth")

### Post-Simulation Visualization

In [ ]:
plt.figure(figsize=(10, 6))
plt.tripcolor(triang, state_t[:, 0].cpu().numpy(), cmap='Blues')
plt.colorbar(label='Water Surface Elevation (m)')
plt.title(f"Final Predicted Water Level at T={current_time/3600:.1f}h")
plt.axis('equal')
plt.show()